# Context and regressors

Short examples of simulator context, design context, and regression formulas.

In [ ]:
import numpy as np
import superstats as sup

## Simulator context

`simulator_context` sends trial-level inputs directly to the simulator.
In the RDM, `boost_idx` selects the accumulator receiving the drift boost.

In [ ]:
model = sup.Model(
    prior=sup.JointPrior(
        v_base=0.5,
        v_diff=1.5,
        a_base=0.6,
        a_diff=0.0,
        tau=0.2,
        sigma_diff=0.0,
    ),
    simulator=sup.simulation.sample_rdm,
    context_simulator={"boost_idx": np.array([0, 1, 0, 1])},
    simulator_context=("boost_idx",),
    missing=None,
)

sample = model.sample(batch_size=2, num_steps=4)
sample["boost_idx"], sample["choice"]

## Design context and a continuous predictor

`design_context` sends predictors to `Formula`. Here `x` changes the RDM's
drift contrast. Simulator context can be used at the same time.

In [ ]:
model = sup.Model(
    prior=sup.JointPrior(
        v_base=0.5,
        v_0=1.5,
        b_x=0.5,
        a_base=0.6,
        a_diff=0.0,
        tau=0.2,
        sigma_diff=0.0,
    ),
    simulator=sup.simulation.sample_rdm,
    formula=sup.Formula(["v_diff = v_0 + b_x * x"]),
    context_simulator={
        "x": np.array([-1.0, 0.0, 0.5, 1.0]),
        "boost_idx": np.array([0, 1, 0, 1]),
    },
    design_context=("x",),
    simulator_context=("boost_idx",),
    missing=None,
)

sample = model.sample(batch_size=2, num_steps=4)
sample["v_diff"][..., 0]

## A categorical predictor

Declare its levels and reference, then associate each other level with a
coefficient through `C(...)`. Neutral contributes zero.
The following examples use `resolve()` to show the formula results directly.

In [ ]:
categorical = {
    "condition": sup.Categorical(["neutral", "speed", "accuracy"], reference="neutral"),
}

formula = sup.Formula(
    ["v_diff = v_0 + C(condition, speed=b_speed, accuracy=b_accuracy)"],
    categorical=categorical,
)

formula.resolve(
    parameters={"v_0": 1.5, "b_speed": -0.3, "b_accuracy": 0.4},
    context={"condition": np.array(["neutral", "speed", "accuracy"])},
)["v_diff"]

## A dummy predictor without `C()`

A numeric 0/1 predictor still works directly. It does not need a categorical
declaration or `C(...)`.

In [ ]:
formula = sup.Formula(["v_diff = v_0 + b_dummy * dummy"])

formula.resolve(
    parameters={"v_0": 1.5, "b_dummy": 0.4},
    context={"dummy": np.array([0, 1, 0, 1])},
)["v_diff"]

## Continuous × continuous interaction

Multiply the predictors. These examples show the interaction term; add main
effects with `+` when wanted. The same arithmetic works with numeric dummies.

In [ ]:
formula = sup.Formula(["v_diff = v_0 + b_interaction * x * z"])

formula.resolve(
    parameters={"v_0": 1.5, "b_interaction": 0.4},
    context={"x": np.array([-1.0, 0.0, 1.0]), "z": np.array([0.5, 1.0, 0.5])},
)["v_diff"]

## Continuous × categorical interaction

Multiply the continuous predictor by the categorical term. Here speed and
accuracy have their own slope adjustments; the reference adjustment is zero.

In [ ]:
formula = sup.Formula(
    ["v_diff = v_0 + C(condition, speed=b_speed, accuracy=b_accuracy) * x"],
    categorical=categorical,
)

formula.resolve(
    parameters={"v_0": 1.5, "b_speed": -0.3, "b_accuracy": 0.4},
    context={
        "condition": np.array(["neutral", "speed", "accuracy"]),
        "x": np.array([1.0, 0.5, 1.0]),
    },
)["v_diff"]

## Categorical × categorical interaction

Nest categorical terms to give combinations their own coefficients.
Here the condition effect applies only to the treatment group.

In [ ]:
formula = sup.Formula(
    ["v_diff = v_0 + C(group, treatment=C(condition, speed=b_speed, accuracy=b_accuracy))"],
    categorical={
        **categorical,
        "group": sup.Categorical(["control", "treatment"], reference="control"),
    },
)

formula.resolve(
    parameters={"v_0": 1.5, "b_speed": -0.3, "b_accuracy": 0.4},
    context={
        "group": np.array(["control", "treatment", "treatment"]),
        "condition": np.array(["speed", "speed", "accuracy"]),
    },
)["v_diff"]